In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
import numpy as np

def build_oracle(distances, guess, num_qubits, qr):
    """Oracle: phase flip for all indices with distance < guess"""
    qc = QuantumCircuit(qr)
    for i, d in enumerate(distances):
        if d < guess:
            bits = format(i, f'0{num_qubits}b')[::-1]
            for idx, b in enumerate(bits):
                if b == '0': qc.x(qr[idx])
            if num_qubits == 1:
                qc.z(qr[0])
            else:
                qc.h(qr[-1])
                qc.mcx(list(range(num_qubits-1)), qr[-1])
                qc.h(qr[-1])
            for idx, b in enumerate(bits):
                if b == '0': qc.x(qr[idx])
    return qc.to_gate(label="Oracle")

def grover_diffusion(num_qubits, qr):
    """Standard Grover diffusion operator"""
    qc = QuantumCircuit(qr)
    qc.h(qr)
    qc.x(qr)
    if num_qubits == 1:
        qc.z(qr[0])
    else:
        qc.h(qr[-1])
        qc.mcx(list(range(num_qubits-1)), qr[-1])
        qc.h(qr[-1])
    qc.x(qr)
    qc.h(qr)
    return qc.to_gate(label="Diffusion")

def quantum_k_min_search(distances, k, shots=1024):
    """Fully quantum k-min search using Grover on AerSimulator"""
    n = len(distances)
    num_qubits = int(np.ceil(np.log2(n)))
    minima_indices = []
    remaining = distances.copy()
    
    sim = AerSimulator()
    qr = QuantumRegister(num_qubits)
    cr = ClassicalRegister(num_qubits)
    
    for _ in range(k):
        guess = max(remaining) + 1
        converged = False
        while not converged:
            qc = QuantumCircuit(qr, cr)
            qc.h(qr)
            qc.append(build_oracle(remaining, guess, num_qubits, qr), qr)
            qc.append(grover_diffusion(num_qubits, qr), qr)
            qc.measure(qr, cr)
            
            result = sim.run(transpile(qc, sim), shots=shots).result()
            counts = result.get_counts()
            
            # Pick most probable index
            i_new = int(max(counts, key=counts.get), 2)
            if remaining[i_new] < guess:
                guess = remaining[i_new]
            else:
                converged = True
        
        minima_indices.append(i_new)
        remaining[i_new] = float('inf')  # exclude found minimum
    
    minima_values = [distances[i] for i in minima_indices]
    return minima_indices, minima_values


In [5]:
distances = [5, 2, 7, 1, 6, 3, 4, 0]
k = 3

indices, values = quantum_k_min_search(distances, k)
print("Indices of k smallest distances:", indices)
print("Corresponding distances:", values)


Indices of k smallest distances: [6, 6, 6]
Corresponding distances: [4, 4, 4]
